In [10]:
# Подключение Google-диска
from google.colab import drive
drive.mount('/content/drive')
!wget http://download.cdn.yandex.net/mystem/mystem-3.0-linux3.1-64bit.tar.gz
!tar -xvf mystem-3.0-linux3.1-64bit.tar.gz
!mv mystem /usr/local/bin/mystem
!chmod +x /usr/local/bin/mystem

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--2025-06-29 14:14:11--  http://download.cdn.yandex.net/mystem/mystem-3.0-linux3.1-64bit.tar.gz
Resolving download.cdn.yandex.net (download.cdn.yandex.net)... 37.9.64.225, 2a02:6b8:23::225
Connecting to download.cdn.yandex.net (download.cdn.yandex.net)|37.9.64.225|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: http://cloudcdn-m9-1.cdn.yandex.net/download.cdn.yandex.net/mystem/mystem-3.0-linux3.1-64bit.tar.gz?lid=235 [following]
--2025-06-29 14:14:12--  http://cloudcdn-m9-1.cdn.yandex.net/download.cdn.yandex.net/mystem/mystem-3.0-linux3.1-64bit.tar.gz?lid=235
Resolving cloudcdn-m9-1.cdn.yandex.net (cloudcdn-m9-1.cdn.yandex.net)... 37.9.111.196, 2a02:6b8:c35:4:0:562:0:20
Connecting to cloudcdn-m9-1.cdn.yandex.net (cloudcdn-m9-1.cdn.yandex.net)|37.9.111.196|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16

# **С пунктуацией**

In [11]:
import os
import pickle
from tqdm import tqdm
import re
from pymystem3 import Mystem

#Функция для предобработки текстов
myst = Mystem()

def preprocess_text(text, lemmatize=False):
    text = re.sub('((www\.[^\s]+)|(https?://[^\s]+))','URL', text)
    text = re.sub('@[^\s]+','USER', text)
    text = text.lower().replace("ё", "е")
    #text = re.sub('[^a-zA-Zа-яА-Я]+', ' ', text)  #исключаем все знаки препинания из модели
    text = re.sub('([\(\:\)]+)', ' \g<1> ', text)   #отделяем смайлики (:) от текста для учета в модели
    text = re.sub('[^a-zA-Zа-яА-Я\(\:\)]+', ' ', text) # оставляем смайлы в текста для учета в модели
    text = re.sub(' +',' ', text)
    text = text.strip()

    if lemmatize:
        return ''.join(myst.lemmatize(text)).strip()
    return text

def tokenize_func(texts):
    return texts.split(' ')

#Загрузка модели (с пунктуацией) и векторизаторов
path_to_models = 'drive/MyDrive/SFU 3/Ex ml/4_сентимент_анализ/Sentiment_analysis_models/with_punct'
path_to_vectori = 'drive/MyDrive/SFU 3/Ex ml/4_сентимент_анализ/Sentiment_analysis_models/with_punct'

with open(os.path.join(path_to_models, 'clf_with_punct.pkl'), 'rb') as f:
    clf = pickle.load(f)

with open(os.path.join(path_to_vectori, 'count_vectorizer.pkl'), 'rb') as f:
    count_vect = pickle.load(f)

with open(os.path.join(path_to_vectori, 'tfidf_transformer.pkl'), 'rb') as f:
    tfidf_transformer = pickle.load(f)

#Чтение текстов
def read_texts_several(folder):
    texts = []
    files = os.listdir(folder)
    for file in files:
        with open(os.path.join(folder, file), 'r', encoding='utf8') as f:
            texts.append(f.read())
    return texts, files

texts, texts_names = read_texts_several('drive/MyDrive/SFU 3/Ex ml/1_подготовленный_корпус/corpus')

#Предобработка
preprocessed = [preprocess_text(text, lemmatize=False) for text in tqdm(texts)]

#Векторизация
X_counts = count_vect.transform(preprocessed)
X_tfidf = tfidf_transformer.transform(X_counts)

# Предсказание
labels = clf.predict(X_tfidf)
labels

100%|██████████| 98/98 [00:00<00:00, 680.24it/s]


array([ 1,  1,  1,  1, -1,  1,  1,  1, -1,  1, -1, -1, -1,  1, -1, -1, -1,
        1, -1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1,  1,  1,  1,
        1,  1,  1,  1,  1, -1,  1,  1,  1,  1,  1,  1, -1,  1, -1, -1,  1,
        1,  1,  1,  1, -1,  1,  1, -1, -1,  1,  1,  1, -1,  1,  1, -1,  1,
        1,  1,  1,  1,  1, -1,  1,  1,  1,  1,  1, -1,  1,  1, -1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1])

# **Сохранение в папки**

In [12]:
def save_to_folder(texts, names, path_to_save, delim='\n'):
    if not os.path.exists(path_to_save):
        os.makedirs(path_to_save)
    for i in range(len(texts)):
        with open(os.path.join(path_to_save, names[i]), 'w', encoding='utf8') as f:
            text = texts[i].replace(delim, ' ')
            f.write(text)

def save_files(texts, labels, texts_names, path_to_save, delim='\n'):
    positive_texts = []
    positive_names = []
    negative_texts = []
    negative_names = []

    for i in range(len(labels)):
        if labels[i] == 1:
            positive_texts.append(texts[i])
            positive_names.append(texts_names[i])
        else:
            negative_texts.append(texts[i])
            negative_names.append(texts_names[i])

    save_to_folder(positive_texts, positive_names, os.path.join(path_to_save, 'positive'), delim)
    save_to_folder(negative_texts, negative_names, os.path.join(path_to_save, 'negative'), delim)

save_files(
    texts,
    labels,
    texts_names,
    path_to_save = 'drive/My Drive/SFU 3/Ex ml/4_сентимент_анализ/На_модели_с_пунктуацией/',
    delim = '\n'
)

# **Без пунктуации**

In [13]:
import os
import pickle
from tqdm import tqdm
import re
from pymystem3 import Mystem

#Функция для предобработки текстов
myst = Mystem()

def preprocess_text_no_punct(text, lemmatize=False):
    text = re.sub('((www\.[^\s]+)|(https?://[^\s]+))','URL', text)
    text = re.sub('@[^\s]+','USER', text)
    text = text.lower().replace("ё", "е")
    text = re.sub('[^a-zA-Zа-яА-Я]+', ' ', text)  #исключаем все знаки препинания из модели
    text = re.sub('([\(\:\)]+)', ' \g<1> ', text)   #отделяем смайлики (:) от текста для учета в модели
    text = re.sub('[^a-zA-Zа-яА-Я\(\:\)]+', ' ', text) # оставляем смайлы в текста для учета в модели
    text = re.sub(' +',' ', text)
    text = text.strip()

    if lemmatize:
        return ''.join(myst.lemmatize(text)).strip()
    return text

#Загрузка модели (без пунктуации) и векторизаторов
path_to_models = 'drive/MyDrive/SFU 3/Ex ml/4_сентимент_анализ/Sentiment_analysis_models/wo_punct'
path_to_vectori = 'drive/MyDrive/SFU 3/Ex ml/4_сентимент_анализ/Sentiment_analysis_models/wo_punct'

with open(os.path.join(path_to_models, 'clf_wo_punct.pkl'), 'rb') as f:
    clf_no_punct = pickle.load(f)

with open(os.path.join(path_to_vectori, 'count_vectorizer_no_punct.pkl'), 'rb') as f:
    count_vect_no_punct = pickle.load(f)

with open(os.path.join(path_to_vectori, 'tfidf_transformer_no_punct.pkl'), 'rb') as f:
    tfidf_transformer_no_punct = pickle.load(f)

#Чтение текстов
def read_texts_several(folder):
    texts = []
    files = os.listdir(folder)
    for file in files:
        with open(os.path.join(folder, file), 'r', encoding='utf8') as f:
            texts.append(f.read())
    return texts, files

texts, texts_names = read_texts_several('drive/MyDrive/SFU 3/Ex ml/1_подготовленный_корпус/corpus')

#Предобработка
texts_chek, texts_names = read_texts_several(folder = 'drive/MyDrive/SFU 3/Ex ml/1_подготовленный_корпус/corpus')
preprocessed_chekhov = [preprocess_text_no_punct(text) for text in texts_chek]

#Векторизация
X_chek_counts = count_vect_no_punct.transform(preprocessed_chekhov)
X_chek_tfidf = tfidf_transformer_no_punct.transform(X_chek_counts)

# Предсказание
labels_chek = clf_no_punct.predict(X_chek_tfidf)
labels


array([ 1,  1,  1,  1, -1,  1,  1,  1, -1,  1, -1, -1, -1,  1, -1, -1, -1,
        1, -1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1,  1,  1,  1,
        1,  1,  1,  1,  1, -1,  1,  1,  1,  1,  1,  1, -1,  1, -1, -1,  1,
        1,  1,  1,  1, -1,  1,  1, -1, -1,  1,  1,  1, -1,  1,  1, -1,  1,
        1,  1,  1,  1,  1, -1,  1,  1,  1,  1,  1, -1,  1,  1, -1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1])

# **Сохранение в папки**

In [14]:
def save_to_folder(texts, names, path_to_save, delim='\n'):
    if not os.path.exists(path_to_save):
        os.makedirs(path_to_save)
    for i in range(len(texts)):
        with open(os.path.join(path_to_save, names[i]), 'w', encoding='utf8') as f:
            text = texts[i].replace(delim, ' ')
            f.write(text)

def save_files(texts, labels, texts_names, path_to_save, delim='\n'):
    positive_texts = []
    positive_names = []
    negative_texts = []
    negative_names = []

    for i in range(len(labels)):
        if labels[i] == 1:
            positive_texts.append(texts[i])
            positive_names.append(texts_names[i])
        else:
            negative_texts.append(texts[i])
            negative_names.append(texts_names[i])

    save_to_folder(positive_texts, positive_names, os.path.join(path_to_save, 'positive'), delim)
    save_to_folder(negative_texts, negative_names, os.path.join(path_to_save, 'negative'), delim)

save_files(
    texts_chek,
    labels_chek,
    texts_names,
    path_to_save = 'drive/My Drive/SFU 3/Ex ml/4_сентимент_анализ/На_модели_без_пунктуации/',
    delim = '\n'
)

# **Сравнение результатов**

In [9]:
import numpy as np

with_punct = np.array([ 1,  1,  1,  1, -1,  1,  1,  1, -1,  1, -1, -1, -1,  1, -1, -1, -1,
        1, -1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1,  1,  1,  1,
        1,  1,  1,  1,  1, -1,  1,  1,  1,  1,  1,  1, -1,  1, -1, -1,  1,
        1,  1,  1,  1, -1,  1,  1, -1, -1,  1,  1,  1, -1,  1,  1, -1,  1,
        1,  1,  1,  1,  1, -1,  1,  1,  1,  1,  1, -1,  1,  1, -1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1])

wo_punct = np.array([ 1,  1,  1,  1, -1,  1,  1,  1, -1,  1, -1, -1, -1,  1, -1, -1, -1,
        1, -1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1,  1,  1,  1,
        1,  1,  1,  1,  1, -1,  1,  1,  1,  1,  1,  1, -1,  1, -1, -1,  1,
        1,  1,  1,  1, -1,  1,  1, -1, -1,  1,  1,  1, -1,  1,  1, -1,  1,
        1,  1,  1,  1,  1, -1,  1,  1,  1,  1,  1, -1,  1,  1, -1,  1,  1,
        1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1, -1])

diff_indices = np.where(with_punct != wo_punct)[0]
print("Индексы различий:", diff_indices)
print("Количество различий:", len(diff_indices))

Индексы различий: []
Количество различий: 0


# **Вывод**

Оба классификатора показали одинаковые результаты: все предсказания по
каждому тексту совпали. Это означает, что в случае моего корпуса художественных текстов пунктуация не оказала влияния на итоговую классификацию.